# Markov Chains

---
In class today we will be implementing a Markov chain to process sentences

---
## Learning Objectives

1. Students will be able to explain the Markov Chain process
1. Implement a Markov Chain


Markov Chains represent a series of events following the Markov Property: future states are memory-less in that they depend only on the current state. This can be expanded to the idea of variable order Markov models where there is a variable-length memory (eg. 1st order Markov Model). Markov models consist of fully observable states. 

> A common example of this is in predicting the weather: We can clearly see the current weather and would like to predict tomorrow's weather. This is also applicable to biology with one case being CpG islands. 

Our goal today will be to implement a Markov model built from words. For our example text, we will use the classic example of Dr. Seuss because of the repetitive nature of the text.

---
## Train Markov model

For our initial implementation of the Markov Model, we will use the simple example of Dr. Seuss: "One fish two fish red fish blue fish."



In [ ]:
def build_markov_model(markov_model, new_text):
    '''
    Function to build or add to a 1st order Markov model given a string of text
    We will store the markov model as a dictionary of dictionaries
    The key in the outer dictionary represents the current state
    and the inner dictionary represents the next state with their contents containing
    the transition probabilities.
    Note: This would be easier to read if we were to build a class representation
           of the model rather than a dictionary of dictionaries, but for simplicitiy
           our implementation will just use this structure.
    
    Args: 
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
        new_text (str): a string to build or add to the moarkov_model

    Returns:
        markov_model (dict of dicts): an updated markov_model
        
    Pseudocode:
        # Step 1: Split text into a list of words
        words = split new_text on whitespace

        # Step 2: Add artificial start and end states
        words = ['*S*'] + words + ['*E*']

        # Step 3: Walk through consecutive word pairs
        FOR i FROM 0 TO length(words) - 2:
            current_state = words[i]
            next_state    = words[i + 1]

            # Step 4: Ensure current_state exists as a key in markov_model
            IF current_state NOT IN markov_model:
                markov_model[current_state] = empty dict

            # Step 5: Ensure next_state exists as a key in the inner dict
            IF next_state NOT IN markov_model[current_state]:
                markov_model[current_state][next_state] = 0

            # Step 6: Increment the transition count
            markov_model[current_state][next_state] += 1

        # Step 7: Return the updated model
        RETURN markov_model    
    '''

    words = new_text.split()
    words = ['*S*'] + words + ['*E*']  
    for i in range(len(words) - 1):
        current_word = words[i]
        next_word = words[i + 1]
        if current_word not in markov_model:
            markov_model[current_word] = {}
        if next_word not in markov_model[current_word]:
            markov_model[current_word][next_word] = 0
        markov_model[current_word][next_word] += 1
    return markov_model


In [92]:
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text)
print (markov_model)

{'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}


###  Nth order Markov chain
In the above model, each event or word is output from only the previous state with no memory of any prior states. While this is useful in some cases, typical biological applications of Markov chains require higher-order models to accurately capture what we know about a system. For instance, in attempting to identify coding regions of a genome, we know that open reading frames (ORFs) contain codon triplets, and so a third or sixth order Markov chain would better describe these regions. Here you will implement a generalized form of our previous Markov Chain to allow for Nth order chains.


In [ ]:
def build_markov_model(markov_model, text, order=1):
    '''
    Function to build or add to a Nth order Markov model given a string of text

    Args: 
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
            or None if a new model is being built
        new_text (str): a string to build or add to the moarkov_model
        order (int): the number of previous states to consider for the model
        
    Returns:
        markov_model (dict of dicts): an updated/new markov_model

    Psudeocode
    Step 1: If markov_model is None, set it to an empty dictionary
    IF markov_model is None:
        markov_model = empty dictionary

    Step 2: Split text into a list of words
    words = split text on whitespace
    
    Step 3: add order copies of '*S*' and one '*E*' to signify start and stop
    words = ['*S'] * order + words + ['*E*']

    Step 4: Walk through the text using a sliding window of size order + 1
    For i From 0 to length(words) - order - 1:
        current_state = tuple(words[i : i + order])
        next_state = words[i + order]
        
        Step 5: Ensure current_state exists as a key in markov_model
        If current state Not In markov_model:
            markov_model[current_state] = empty dictionary
        
        Step 6: Ensure next_state exists as a key in the inner dictionary
        If next_state Not In markov_model[current_state]:
            markov_model[current_state][next_state] = 0
        
        Step 7: Increment the transition count
        markov_model[current_state][next_state] += 1
    
    Step 8: Return the updated model
    Return markov_model 
    '''
    if markov_model is None:
        markov_model = {}

    words = text.split()
    words = ['*S*'] * order + words + ['*E*']

    for i in range(len(words) - order):
        current_state = tuple(words[i:i + order])
        next_state = words[i + order]

        if current_state not in markov_model:
            markov_model[current_state] = {}
        if next_state not in markov_model[current_state]:
            markov_model[current_state][next_state] = 0
        markov_model[current_state][next_state] += 1
    return markov_model

In [94]:
markov_model = dict()
text = "one fish two fish red fish blue red fish blue"
markov_model = build_markov_model(markov_model, text, order=2)
markov_model

{('*S*', '*S*'): {'one': 1},
 ('*S*', 'one'): {'fish': 1},
 ('one', 'fish'): {'two': 1},
 ('fish', 'two'): {'fish': 1},
 ('two', 'fish'): {'red': 1},
 ('fish', 'red'): {'fish': 1},
 ('red', 'fish'): {'blue': 2},
 ('fish', 'blue'): {'red': 1, '*E*': 1},
 ('blue', 'red'): {'fish': 1}}

## Generate text from Markov Model

Markov models are "generative models". That is, the probability states in the model can be used to generate output following the conditional probabilities in the model.

We will now generate a sequence of text from the Markov model. For this section, I recommend using np.random.choice, which allows for you to provide a probability distribution for drawing the next edge in the chain.

In [ ]:
import numpy as np

text = "one fish two fish red fish blue red fish blue"

def get_next_word(current_word, markov_model, seed=42):
    '''
    Function to randomly move a valid next state given a markov model
    and a current state (word)
    
    Args: 
        current_word (tuple): a word that exists in our model
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)

    Returns:
        next_word (str): a randomly selected next word based on transition probabilies
        
    Pseudocode:
    Step 1: Look up the possible next words and their counts for current_word
    possible_wordsd = markov_model[current_word]

    Step 2: Calculate the total count of all possible next words
    total_count = sum of possible_words' valyes

    Step 3: Calculate the transition probability for each next word
    For each word, count In possible_words:
        probabilities[word] = count / total_count

    Step 4: Extract the words and their matching probabilities as two aligned lists
    words_list = list of probabilities' keys
    probs_list = list of probabilities' values

    Step 5: Randomly select the next word, weighted by its probability
    Set random seed to seed
    next_word = weighted random choice from words_list using probs_list as weights

    Step 6: return the selected next word
    Return next_word
        
    '''
    probabilities = {}
    possible_words = markov_model[current_word]
    total_count = sum(possible_words.values())
    for word, count in possible_words.items():
        probabilities[word] = count / total_count
    words_list = list(probabilities.keys())
    probs_list = list(probabilities.values())
    np.random.seed(seed)
    next_word = np.random.choice(words_list, p=probs_list)
    
    return next_word

def generate_random_text(markov_model, seed=42):
    '''
    Function to generate text given a markov model
    
    Args: 
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)

    Returns:
        sentence (str): a randomly generated sequence given the model
        
    Pseudocode:
    Step 1: Determine the order of the model by checking the length of an arbitrary key
    some_key = any key from markov_model
    order = length of some_key

    Step 2: Initialize the current state at the start marker(s)
    current_word = tuple of the order copies of '*S*'ArithmeticError

    Step 3: Get the first word using get_next_word, passing along the seed
    next_word = get_next_word(current_word, markov_model, seed)

    Step 4: Repeat until the end marker is generated or a safety limit is reached
    sentence = empty list
    count = 0
    While next_word is not '*E*'
        
        Step 5: Add the generated word to the sentence
        Append next_word to sentence

        Step 6: Slide the window forward using the new word
        current_wrod = current_word[1:] + (next_word,)

        Step 7: Get the next word, again passing along the seed
        next_word = get_next_word(current_word, markov_model, seed)

        Step 8: Stop early if generation runs unexpectedly long
        count += 1
        If count > 2000:
            BREAK

        Step 9: Join the generated words into a signle space-separated string
        sentence = join words with spaces

        Step 10: REturn the final sentence
        Return sentence
        
    '''
    sentence = []
    for key in markov_model:
        some_key = key
        break
    order = len(some_key)
    current_word = tuple(['*S*'] * order)
    next_word = get_next_word(current_word,markov_model,seed)
    count = 0
    while next_word != '*E*':
        sentence.append(next_word)
        current_word = (current_word[1:] + (next_word,))
        next_word = get_next_word(current_word, markov_model, seed)
        count += 1
        if count > 2000:
            break
    sentence = ' '.join(sentence)

    return sentence

---

## All the Fish
Up till now, you have only been working with a line or two of the Dr. Seuss' _One Fish, Two Fish_. Now, I want you to build a model using the whole book and try different orders of Markov models.

> **Pro-tip**: Consider how you signify the beginning of the book, beginning of a line, end of a line, and end of the book.

In [96]:
# Now just add some more training data to the markov model. You can find it under data/one_fish_two_fish.txt

markov_model = dict()
line_count = 0
with open('one_fish_two_fish.txt', "r") as file: 
    for line in file:
        markov_model = build_markov_model(markov_model, line, order=3)
        line_count = line_count + 1 

print (generate_random_text(markov_model,seed=7))

Some are old and some are blue.


---
## Pick Your Poison
There are three texts provided for under `data/`. The first is:
1. Dr. Seuss' "One Fist, Two Fish" (179 lines of text)
2. All of Shakespeare's sonnets (2308 lines of text)
3. Homer's "The Odyssey" (9255 Lines of text)

In [97]:
# An example of a more complex text that we can use to generate more complex output
nth_order_markov_model = dict()
poison_markov_model = dict()
corpus = ''
with open("data/sonnets.txt", "r") as poison_text:
    for line in poison_text:
        if not line.strip():
            nth_order_markov_model = build_markov_model(poison_markov_model, corpus, order=2)
            poison_markov_model = nth_order_markov_model
            corpus = ''
        else:
            corpus += line
nth_order_markov_model = build_markov_model(poison_markov_model, corpus, order=4)
# Process the lines. Consider that sonnets are separated by an empty line.
poison_markov_model = nth_order_markov_model
#print (generate_random_text(poison_markov_model,seed=7))
result = generate_random_text(poison_markov_model, seed=7)
print(result)
print(len(result.split()))

When forty winters shall besiege thy brow, And dig deep trenches in thy glass and tell the face thou viewest Now is the time that face should form another; Whose fresh repair if now thou not renewest, Thou dost beguile the world, or else this glutton be, To eat the world's fresh ornament, And only herald to the very same And that unfair which fairly doth excel; For never-resting time leads summer on To hideous winter, and confounds him there; Sap checked with frost, and lusty leaves quite gone, Beauty o'er-snowed and bareness every where: Then were not summer's distillation left, A liquid prisoner pent in walls of glass, Beauty's effect with beauty were bereft, Nor it, nor no remembrance what it was: But flowers distill'd, though they with winter meet, Leese but their show; their substance still lives sweet.
140
